In [ ]:
# Change path
import os, sys
repo_path = os.path.abspath(os.path.join(os.path.abspath(''), '../..'))
assert os.path.basename(repo_path) == 'textdd', "Wrong parent folder. Please change to 'textdd'"
if sys.path[0] != repo_path:
    sys.path.insert(0, repo_path)

import torch
from datasets import Dataset, DatasetDict, load_dataset
from transformers import AutoTokenizer
import pprint

from expts.expt_utils import get_dataset
from utils.metadata import DatasetMetadata

pprint = pprint.PrettyPrinter(indent=1, width=120, compact=True, sort_dicts=False)

In [ ]:
# List of supported datasets
supported = DatasetMetadata.supported

DATASET_PATHS = {
    "agnews": {"path": "fancyzhx/ag_news"},
    "imdb": {"path": "stanfordnlp/imdb"},
    "sst2": {"path": "stanfordnlp/sst2"},
    "mnlim": {"path": "glue", "name": "mnli"},
    "qqp": {"path": "glue", "name": "qqp"},
    "qnli": {"path": "glue", "name": "qnli"},
}

def get_raw_dataset(dataset: str) -> DatasetDict:
    dataset: DatasetDict = load_dataset(
        path=DATASET_PATHS[dataset]["path"], 
        name=DATASET_PATHS[dataset].get("name", None),
        cache_dir="../../datasets",
    )
    return dataset

def config_dataset_small_factory(dataset: str, unify_text: bool = False) -> dict:
    if dataset not in supported:
        raise ValueError(f"Unsupported dataset {dataset}. Supported datasets: {supported}")

    config = {
        "abbrev": dataset,
        "splits": ["train"],
        "train": {
            "init_with": "load_dataset",
            "load_dataset_kwargs": {
                "path": DATASET_PATHS[dataset]["path"],
                "name": DATASET_PATHS[dataset].get("name", None),
                "cache_dir": "../../datasets",
                "split": "train[:1%]",
            },
            "unify_text": unify_text,
        },
    }
    return config

In [ ]:
dataset = get_raw_dataset("qnli")

In [ ]:
print(dataset)

In [ ]:
def print_raw_metadata(dataset_abbrev:str):
    config = config_dataset_small_factory(dataset_abbrev)
    dataset: DatasetDict = get_dataset(config)
    dataset: Dataset = dataset["train"]
    print(f"Dataset: {dataset_abbrev}")
    # Get metadata of train set
    print(f"1% N samples: {len(dataset)}")
    print(f"Train set features:\n{pprint.pformat(dataset.features)}")
    print(f"Sample 0:\n{pprint.pformat(dataset[0])}")

print_raw_metadata("qnli")

In [ ]:
def print_preprocessed_samples(dataset_abbrev:str):
    config = config_dataset_small_factory(dataset_abbrev, unify_text=True)
    dataset: DatasetDict = get_dataset(config)
    dataset: Dataset = dataset["train"]
    print(f"Dataset: {dataset_abbrev}")
    # Get metadata of train setconfig_dataset_small_factory
    print(f"1% N samples: {len(dataset)}")
    print(f"Train set features:\n{pprint.pformat(dataset.features)}")
    print(f"Sample 0:\n{pprint.pformat(dataset[0])}")

print_preprocessed_samples("qnli")

In [ ]:
from collections import Counter
from nltk.tokenize import wordpunct_tokenize
import numpy as np
import tqdm.auto as tqdm

def get_stats_words_count(dataset: str, tokenizer='wordpunct', split: str = "train"):
    ds = get_raw_dataset(dataset=dataset)
    if dataset == "mnlim":
        ds = ds.map(lambda x: {"text": x["premise"] + " " + x["hypothesis"]})
    elif dataset == "qqp":
        ds = ds.map(lambda x: {"text": x["question1"] + " " + x["question2"]})
    elif dataset == "sst2":
        ds = ds.map(lambda x: {"text": x["sentence"]})
    elif dataset == "qnli":
        ds = ds.map(lambda x: {"text": x["question"] + " " + x["sentence"]})


    counter = Counter()
    lengths = []
    for example in tqdm.tqdm(ds["train"]):
        if tokenizer == 'wordpunct':
            tokens = wordpunct_tokenize(example["text"])
        else:
            tokens = tokenizer(example["text"])["input_ids"]
        counter.update(tokens)
        lengths.append(len(tokens))
    lengths = np.array(lengths)
    
    mean = lengths.mean()
    std = lengths.std()
    quantiles = {q: int(np.percentile(lengths, q)) for q in [0, 10, 25, 50, 75, 90, 100]}
    print(f"Dataset: {dataset}, split: {split}")
    print(f"""
        "mean": {mean:.2f},
        "std": {std:.2f},
        "quantiles": {quantiles},
    """)

tokenizer = AutoTokenizer.from_pretrained(
    pretrained_model_name_or_path="google/gemma-3-270m",
    cache_dir="../../pretrained/llms/",
)
get_stats_words_count("qnli", tokenizer=tokenizer)